# Custom Estimators

1. **Estimator** = any object in scikit-learn that implements a `.fit(X, y)` method (learns something from data). Most estimators also implement `.predict()`, `.transform()`, or both.

2. **Custom Estimator** = a class you build yourself, following sklearn's conventions, when the built-in estimators don't do what you need. You're free to define whatever logic you want inside `.fit()` and `.predict()`.

3. **Why inherit from `BaseEstimator` and Mixins:**
   - `BaseEstimator` → gives you `get_params()` and `set_params()` for free. This is **required** for your estimator to work with `GridSearchCV`/`RandomizedSearchCV` (hyperparameter tuning) and with `sklearn.clone()`.
   - `ClassifierMixin` → gives you a default `.score()` method (accuracy) for classifiers.
   - `RegressorMixin` → gives you a default `.score()` method (R²) for regressors.
   - `TransformerMixin` → gives you `.fit_transform()` for free, built automatically from your `.fit()` + `.transform()`.

In [8]:
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np
from sklearn.utils import check_X_y

# Custom Estimator

In [9]:
class MostFrequentClassClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self):
        self.most_frequent_ = None

    def fit(self, X, y):

        # Validate input X and target vector y
        X, y = check_X_y(X, y)

        # Ensure y is 1D
        y = np.ravel(y)

        # Manually compute the most frequent class
        unique_classes, counts = np.unique(y, return_counts=True)
        self.most_frequent_ = unique_classes[np.argmax(counts)]

        return self

    def predict(self, X):
        if self.most_frequent_ is None:
            raise ValueError("This classifier instance is not fitted yet.")
        # Predict the most frequent class for each input sample
        return np.full(shape=(X.shape[0],), fill_value=self.most_frequent_)


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

# Load data
iris = load_iris()
X, y = iris.data, iris.target

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Initialize and fit the custom estimator
classifier = MostFrequentClassClassifier()
classifier.fit(X_train, y_train)

# Make predictions
predictions = classifier.predict(X_test)

# Evaluate the custom estimator
print(f"Predicted class for all test instances: {predictions[0]}")


Predicted class for all test instances: 1


In [11]:
# due to classifier mixin I am able to impement score function automatically.
classifier.score(X_test,y_test)

0.2894736842105263

In [12]:
classifier.most_frequent_

np.int64(1)

In [13]:
from sklearn.model_selection import cross_val_score

cross_val_score(classifier, X_train, y_train)

array([0.34782609, 0.26086957, 0.27272727, 0.18181818, 0.31818182])